In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 200)
print("Imports OK")

Imports OK


In [2]:
df = pd.read_csv("train_data_clean.csv")
print("Loaded: train_data_clean.csv")
print("Shape:", df.shape)
display(df.head())
display(df.dtypes.to_frame("dtype"))

Loaded: train_data_clean.csv
Shape: (53991, 9)


,Date of Sale (dd/mm/yyyy),Sale Year,Sale Month,is_apartment,location,Description of Property,Not Full Market Price,Price_Adjusted_VAT_Clamped,vat_adjusted_flag
0,2016-09-30,2016,9,0,Cork,Second-Hand Dwelling house /Apartment,No,181000.00000,0
1,2016-12-20,2016,12,0,Cork,New Dwelling house /Apartment,No,60000.00000,1
2,2016-09-28,2016,9,1,Wexford,New Dwelling house /Apartment,No,70565.00435,1
3,2016-09-16,2016,9,0,Wicklow,New Dwelling house /Apartment,No,253499.98000,1
4,2016-01-29,2016,1,0,Dublin 27,Second-Hand Dwelling house /Apartment,No,310000.00000,0


,dtype
Date of Sale (dd/mm/yyyy),str
Sale Year,int64
Sale Month,int64
is_apartment,int64
location,str
Description of Property,str
Not Full Market Price,str
Price_Adjusted_VAT_Clamped,float64
vat_adjusted_flag,int64


In [3]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

target = "Price_Adjusted_VAT_Clamped"

num_features = ["Sale Year", "Sale Month", "is_apartment", "vat_adjusted_flag"]
cat_features  = ["location", "Description of Property", "Not Full Market Price"]

X_num = df[num_features].copy()
X_cat = df[cat_features].copy()
y = df[target].copy()

encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
X_cat_encoded = encoder.fit_transform(X_cat)
cat_col_names = encoder.get_feature_names_out(cat_features)

X = pd.concat([
    X_num.reset_index(drop=True),
    pd.DataFrame(X_cat_encoded, columns=cat_col_names)
], axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"X shape: {X.shape}")
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

X shape: (53991, 60)
Train: (43192, 60), Test: (10799, 60)


In [4]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1        # 使用所有CPU核心，加快训练
)

print("Training Random Forest...")
rf_model.fit(X_train, y_train)
print("Done!")

# --- Metrics ---
y_pred_train = rf_model.predict(X_train)
y_pred_test  = rf_model.predict(X_test)

rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
rmse_test  = np.sqrt(mean_squared_error(y_test,  y_pred_test))
r2_train   = r2_score(y_train, y_pred_train)
r2_test    = r2_score(y_test,  y_pred_test)
mae_test   = mean_absolute_error(y_test, y_pred_test)

print(f"\n{'':20s} {'Train':>12s} {'Test':>12s}")
print(f"{'RMSE':20s} {rmse_train:>12.0f} {rmse_test:>12.0f}")
print(f"{'R²':20s} {r2_train:>12.4f} {r2_test:>12.4f}")
print(f"{'MAE (test)':20s} {'':>12s} {mae_test:>12.0f}")

Training Random Forest...
Done!

                            Train         Test
RMSE                       108898       134513
R²                         0.6109       0.4044
MAE (test)                               98469


In [6]:
rf_model2 = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,       # 限制树的深度
    min_samples_leaf=20, # 每个叶节点至少20个样本
    random_state=42,
    n_jobs=-1
)

print("Training Random Forest (constrained)...")
rf_model2.fit(X_train, y_train)
print("Done!")

y_pred_train2 = rf_model2.predict(X_train)
y_pred_test2  = rf_model2.predict(X_test)

rmse_train2 = np.sqrt(mean_squared_error(y_train, y_pred_train2))
rmse_test2  = np.sqrt(mean_squared_error(y_test,  y_pred_test2))
r2_train2   = r2_score(y_train, y_pred_train2)
r2_test2    = r2_score(y_test,  y_pred_test2)
mae_test2   = mean_absolute_error(y_test, y_pred_test2)

print(f"\n{'':20s} {'Train':>12s} {'Test':>12s}")
print(f"{'RMSE':20s} {rmse_train2:>12.0f} {rmse_test2:>12.0f}")
print(f"{'R²':20s} {r2_train2:>12.4f} {r2_test2:>12.4f}")
print(f"{'MAE (test)':20s} {'':>12s} {mae_test2:>12.0f}")

Training Random Forest (constrained)...
Done!

                            Train         Test
RMSE                       143587       145639
R²                         0.3234       0.3018
MAE (test)                              111798


In [7]:
rf_model3 = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,        # 比默认浅，比10深
    min_samples_leaf=5,  # 适中
    random_state=42,
    n_jobs=-1
)

print("Training Random Forest (max_depth=20)...")
rf_model3.fit(X_train, y_train)
print("Done!")

y_pred_train3 = rf_model3.predict(X_train)
y_pred_test3  = rf_model3.predict(X_test)

rmse_train3 = np.sqrt(mean_squared_error(y_train, y_pred_train3))
rmse_test3  = np.sqrt(mean_squared_error(y_test,  y_pred_test3))
r2_train3   = r2_score(y_train, y_pred_train3)
r2_test3    = r2_score(y_test,  y_pred_test3)
mae_test3   = mean_absolute_error(y_test, y_pred_test3)

print(f"\n{'':20s} {'Train':>12s} {'Test':>12s}")
print(f"{'RMSE':20s} {rmse_train3:>12.0f} {rmse_test3:>12.0f}")
print(f"{'R²':20s} {r2_train3:>12.4f} {r2_test3:>12.4f}")
print(f"{'MAE (test)':20s} {'':>12s} {mae_test3:>12.0f}")

Training Random Forest (max_depth=20)...
Done!

                            Train         Test
RMSE                       128964       134818
R²                         0.4542       0.4017
MAE (test)                              101617


In [8]:
import re

date_col  = "Date of Sale (dd/mm/yyyy)"
price_col = "Price (€)"
VAT_RATE  = 0.135

# 复用训练集的 p05/p95
p05 = df["Price_Adjusted_VAT_Clamped"].quantile(0.05)
p95 = df["Price_Adjusted_VAT_Clamped"].quantile(0.95)

def build_location(row):
    county   = str(row["County"]).strip() if pd.notna(row["County"]) else ""
    district = row["dublin_district"]
    if county == "Dublin":
        if pd.notna(district) and str(district).strip() != "":
            return "Dublin " + str(district).strip()
        else:
            return "Dublin Other"
    else:
        return county if county else "Unknown"

# --- Load ---
df_test_raw = pd.read_csv("test_data.csv")
print("Loaded: test_data.csv, Shape:", df_test_raw.shape)

df_test = df_test_raw.copy()

# --- Text normalization ---
for c in [col for col in df_test.columns if df_test[col].dtype == "object"]:
    df_test[c] = df_test[c].apply(lambda x: re.sub(r"\s+", " ", str(x).strip()) if pd.notna(x) else x)

# --- Date & Price parsing ---
df_test[date_col] = pd.to_datetime(df_test[date_col], dayfirst=True, errors="coerce")
df_test[price_col] = pd.to_numeric(
    df_test[price_col].astype(str).str.replace("€", "", regex=False).str.replace(",", "", regex=False),
    errors="coerce"
)

# --- Time features ---
df_test["Sale Year"]  = df_test[date_col].dt.year
df_test["Sale Month"] = df_test[date_col].dt.month

# --- is_apartment ---
df_test["is_apartment"] = df_test["Address"].str.contains(
    r"apartment|apt|unit|flat", case=False, na=False
).astype(int)

# --- location ---
df_test["location"] = df_test.apply(build_location, axis=1)

# --- Drop duplicates + filter invalid ---
df_test = df_test.drop_duplicates()
reject_mask = (
    df_test[price_col].isna() | (df_test[price_col] <= 0) |
    df_test[date_col].isna() |
    df_test["County"].isna() | (df_test["County"].astype(str).str.strip() == "")
)
df_test = df_test.loc[~reject_mask].copy()

# --- VAT adjustment ---
cond_test = (
    (df_test["Description of Property"].astype(str).str.strip() == "New Dwelling house /Apartment") &
    (df_test["VAT Exclusive"].astype(str).str.strip().str.lower() == "yes") &
    (df_test[price_col].notna())
)
df_test["Price_Adjusted_VAT"] = df_test[price_col].copy()
df_test.loc[cond_test, "Price_Adjusted_VAT"] = df_test.loc[cond_test, price_col] * (1 + VAT_RATE)
df_test["vat_adjusted_flag"] = cond_test.astype(int)

# --- Price clamping (训练集的p05/p95) ---
df_test["Price_Adjusted_VAT_Clamped"] = df_test["Price_Adjusted_VAT"].clip(lower=p05, upper=p95)

print(f"Test shape after cleaning: {df_test.shape}")

Loaded: test_data.csv, Shape: (10000, 14)
Test shape after cleaning: (9999, 21)


In [9]:
# --- Build test features ---
X_test_num = df_test[num_features].copy()
X_test_cat = df_test[cat_features].copy()

X_test_cat_encoded = encoder.transform(X_test_cat)
X_test_cat_df = pd.DataFrame(X_test_cat_encoded, columns=cat_col_names, index=df_test.index)

X_test_final = pd.concat([
    X_test_num.reset_index(drop=True),
    pd.DataFrame(X_test_cat_encoded, columns=cat_col_names)
], axis=1)

y_test_final = df_test["Price_Adjusted_VAT_Clamped"].values

# --- Predict ---
y_pred_rf = rf_model.predict(X_test_final)

# --- Metrics ---
rmse_rf = np.sqrt(mean_squared_error(y_test_final, y_pred_rf))
mae_rf  = mean_absolute_error(y_test_final, y_pred_rf)
r2_rf   = r2_score(y_test_final, y_pred_rf)

print("=== Final Model Comparison (External Test Set) ===")
print(f"\n{'Model':25s} {'R²':>8s} {'RMSE':>12s} {'MAE':>12s}")
print("-" * 60)
print(f"{'Linear Regression':25s} {'0.3976':>8s} {'136,947':>12s} {'104,861':>12s}")
print(f"{'Random Forest':25s} {r2_rf:>8.4f} {rmse_rf:>12,.0f} {mae_rf:>12,.0f}")

=== Final Model Comparison (External Test Set) ===

Model                           R²         RMSE          MAE
------------------------------------------------------------
Linear Regression           0.3976      136,947      104,861
Random Forest               0.2834      149,331      113,972
